In [1]:
import autogen
autogen.__version__

C:\Users\manju\anaconda3\lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")
C:\Users\manju\anaconda3\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Users\manju\anaconda3\lib\site-packages\autogen\oai\gemini.py:44: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README 

'0.2.35'

In [2]:
#!pip install autogen-agentchat[gemini]~=0.2
#!pip install ag2[gemini]
#!pip install chromadb==0.5.1
#!pip install google-generativeai == 0.8.4
#!pip install vertexai == 1.71.1

In [3]:
import sys

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

In [4]:
import logging

logging.getLogger("agentops").setLevel(logging.WARNING)

import agentops

agentops.init("954ecbcb-9555-470b-a2ad-a6999473648b")

🖇 AgentOps: [ConcurrentFuturesInstrumentor] Successfully instrumented concurrent.futures.ThreadPoolExecutor
🖇 AgentOps: You're on the agentops free plan 🤔
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\manju\anaconda3\lib\logging\__init__.py", line 1103, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\manju\anaconda3\lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f914' in position 66: character maps to <undefined>
Call stack:
  File "C:\Users\manju\anaconda3\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\manju\anaconda3\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\manju\anaconda3\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\manju\anaconda3\lib\sit

In [5]:
from pathlib import Path

from autogen import ConversableAgent
from autogen.coding import  CodeBlock, LocalCommandLineCodeExecutor
work_dir = Path("coding")
work_dir.mkdir(exist_ok=True)

executor = LocalCommandLineCodeExecutor(work_dir=work_dir)

#Agent 1
code_executor_agent = ConversableAgent(
    name="code_executor_agent",
    llm_config=False,
    code_execution_config={
        "executor": executor,
    },
    human_input_mode="NEVER",
)

In [6]:
import google.generativeai as genai

print("Google Generative AI imported successfully")
print(genai.__version__)

Google Generative AI imported successfully
0.8.6


In [7]:
import google.generativeai as genai

# Put your Gemini API key here
GEMINI_API_KEY = ""

genai.configure(api_key=GEMINI_API_KEY)

# Create Gemini model
model = genai.GenerativeModel("gemini-3.5-flash")

print("Gemini initialized successfully!")

Gemini initialized successfully!


In [8]:
import uuid

chat_id = str(uuid.uuid4())

print("Chat ID:", chat_id)

Chat ID: e2f38cf3-df47-4d80-86b6-59ea8711dfdd


In [20]:
# The code writer agent's system message is to instruct the LLM on how to
# use the Jupyter code executor with IPython kernel.

import uuid

# Create your own chat ID
chat_id = str(uuid.uuid4())

# Store conversation history
chat_history = []

# Initialize cost information
cost_info = {
    "input_tokens": 0,
    "output_tokens": 0,
    "total_tokens": 0
}


def code_writer(user_task):

    system_message = """
You have been given coding capability to solve tasks using Python code.

In the following cases, suggest Python code in a python coding block
or shell script in a sh coding block for the user to execute.

1. When you need to collect information, use code to output the information
   you need.

2. When you need to perform a task with code, use the code to perform the
   task and output the result.

Solve the task step by step if needed.

If a plan is not provided, explain your plan first.

Be clear which step uses code and which step uses your language skill.

When using code, indicate the script type in the code block.

Do not suggest incomplete code that requires users to modify.

Use print() for relevant output.
"""

    prompt = f"""
{system_message}

USER TASK:
{user_task}
"""

    response = model.generate_content(prompt)

    # Get token usage
    usage = response.usage_metadata

    input_tokens = usage.prompt_token_count
    output_tokens = usage.candidates_token_count
    total_tokens = usage.total_token_count

    # Cost/token information
    cost_info = {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens
    }

    # Store chat history
    chat_history.append({
        "role": "user",
        "content": user_task
    })

    chat_history.append({
        "role": "assistant",
        "content": response.text
    })

    return response.text, cost_info

In [21]:
task = input("Enter your programming task: ")

answer = code_writer(task)

print("\n========== GEMINI RESPONSE ==========\n")
print(answer)

Enter your programming task:   write python code for permutation for the word ALGEbRA. use optimised way to calculate it .I want final result count



========== GEMINI RESPONSE ==========

('Here is the plan to solve the task:\n\n1. **Analyze the input word "ALGEbRA"**: We will count the frequency of each unique character in the word. \n2. **Apply the multiset permutation formula**: The number of unique permutations of a word with duplicate letters is given by:\n   $$\\frac{N!}{n_1! \\times n_2! \\times \\dots \\times n_k!}$$\n   where $N$ is the total number of characters, and $n_i$ represents the frequency of each unique character.\n3. **Write and run Python code** to perform this calculation efficiently using the `math.factorial` function and `collections.Counter`.\n\n### Python Code\n\n```python\nimport math\nfrom collections import Counter\n\ndef calculate_permutations(word):\n    # Count the frequency of each character\n    char_counts = Counter(word)\n    \n    # Calculate N! (numerator)\n    numerator = math.factorial(len(word))\n    \n    # Calculate the product of factorials of character counts (denominator)\n    denomina

In [30]:
task = input("Enter your programming task: ")

answer = code_writer(task)

print("\n========== GEMINI RESPONSE ==========\n")
print(answer)


Enter your programming task:  Write optimized Python code to calculate the number of unique permutations of the word 'ALGEBRA.'     For example, the word 'ALGEbRA' should have 2520 unique permutations. Use efficient logic to compute the result and verify the output using      the word 'ALGEBRA.' The code should only return the final permutation count.



========== GEMINI RESPONSE ==========

("An elegant and optimized way to calculate the number of unique permutations of a word is to use the multinomial coefficient formula:\n\n$$\\frac{n!}{n_1! \\times n_2! \\times \\dots \\times n_k!}$$\n\nWhere:\n- $n$ is the total number of characters in the word.\n- $n_1, n_2, \\dots, n_k$ are the frequencies of each unique character in the word.\n\nTo implement this efficiently in Python, we can use `collections.Counter` to get the frequency of each character and `math.factorial` to perform the factorial calculations. We also convert the word to uppercase to ensure the calculation is case-insensitive.\n\nHere is the complete Python code to solve the task:\n\n```python\nimport math\nfrom collections import Counter\n\ndef unique_permutations_count(word):\n    # Convert the word to uppercase to ensure case insensitivity\n    word = word.upper()\n    \n    # Count the frequency of each character\n    char_counts = Counter(word)\n    \n    # Calculat

In [31]:
dir(answer)

['__add__',
 '__class__',
 '__class_getitem__',
 '__contains__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getnewargs__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__mul__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rmul__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'count',
 'index']

In [32]:
print("\n========== CHAT ID ==========\n")
print(chat_id)


========== CHAT ID ==========

008fb780-ca84-478a-84a6-9bfc3ef9a7b8


In [33]:
print("\n========== CHAT HISTORY ==========\n")

for message in chat_history:
    print("Role:", message["role"])
    print("Content:", message["content"])
    print("----------------------------------")


========== CHAT HISTORY ==========

Role: user
Content:  write python code for permutation for the word ALGEbRA. use optimised way to calculate it .I want final result count
----------------------------------
Role: assistant
Content: Here is the plan to solve the task:

1. **Analyze the input word "ALGEbRA"**: We will count the frequency of each unique character in the word. 
2. **Apply the multiset permutation formula**: The number of unique permutations of a word with duplicate letters is given by:
   $$\frac{N!}{n_1! \times n_2! \times \dots \times n_k!}$$
   where $N$ is the total number of characters, and $n_i$ represents the frequency of each unique character.
3. **Write and run Python code** to perform this calculation efficiently using the `math.factorial` function and `collections.Counter`.

### Python Code

```python
import math
from collections import Counter

def calculate_permutations(word):
    # Count the frequency of each character
    char_counts = Counter(word)
    


In [37]:
# Generate summary
summary_prompt = f"""
Summarize the following conversation briefly.

Conversation:
{chat_history}

Provide:
1. The user's request
2. What was done
3. The final result

Keep the summary concise.
"""

summary_response = model.generate_content(summary_prompt)

summary = summary_response.text

In [38]:
chat_result = {
    "chat_id": chat_id,
    "human_input": task,
    "chat_history": chat_history,
    "cost": cost_info,
    "answer": answer,
    "summary": summary
}

In [39]:
print("\n========== SUMMARY ==========\n")
print(chat_result["summary"])


========== SUMMARY ==========

Here is a concise summary of the conversation:

1. **The User's Request:** 
The user asked for optimized Python code to calculate the number of unique, case-insensitive permutations for the word "ALGEBRA" (specifically matching the behavior of "ALGEbRA") and requested that the code return only the final permutation count.

2. **What Was Done:** 
The assistant provided an efficient Python solution using `collections.Counter` and `math.factorial` to implement the multinomial coefficient formula. The code converts the input word to uppercase to handle case insensitivity, counts the frequency of each character, and divides the factorial of the total word length by the product of the factorials of each character's frequency.

3. **The Final Result:** 
The assistant provided the completed Python code, which outputs the correct final permutation count of **2520** for the word "ALGEBRA".
